# Day 58 — Code review, refactoring, and maintainability
Objectives:
- Refactor notebook code into reusable modules.
- Add unit tests and simple CI checklist.
- Improve documentation (docstrings, README sections).
- Prepare a maintainers checklist for DS repos.

## 1) From notebook to module
Take code from prior days (e.g., your preprocessing + training) and move it into a `src/` package.
Example layout:
````
ds-60day/
  src/
    __init__.py
    data.py        # load/clean functions
    features.py    # feature engineering
    model.py       # train/evaluate/save
  tests/
    test_model.py
  notebooks/
    ...
````


In [ ]:
# Example refactor target
from dataclasses import dataclass
from typing import Tuple
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

@dataclass
class TrainResult:
    pipeline: Pipeline
    auc: float

def build_pipeline() -> Pipeline:
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
                             ('num', StandardScaler(), ['fare','age'])])
    return Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])

def train_evaluate(df: pd.DataFrame) -> TrainResult:
    X = df[['sex','class','fare','age']]
    y = df['survived']
    Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
    pipe = build_pipeline()
    pipe.fit(Xtr,ytr)
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    return TrainResult(pipe, auc)


## 2) Tests with pytest
Create `tests/test_model.py` with unit tests for your functions.
Example:
```python
import pandas as pd
from src.model import train_evaluate

def test_train_evaluate_runs():
    df = pd.DataFrame([
        {
            'survived': i % 2,
            'sex': 'female' if i % 2 else 'male',
            'class': ('First', 'Second', 'Third')[i % 3],
            'fare': 20.0 + i,
            'age': 18.0 + (i % 30),
        }
        for i in range(40)
    ])
    res = train_evaluate(df)
    assert 0.5 <= res.auc <= 1.0
```
Run: `pytest -q`

## 3) Document with docstrings & README
- Ensure every function has a concise docstring (inputs/outputs, assumptions).
- Add a short project README describing how to run training and tests.

## 4) Maintainability checklist
- Style and linting: Ruff; type contracts: mypy
- Tests pass: pytest
- Reproducibility: reviewed lock file, deterministic seeds
- Data contracts: pandera schemas on inputs/outputs
- Logging and error handling
- CI: run pytest + lint on PR

## Learner exercises and progressive hints

1. Move at least two notebook functions into a `src/` package and add tests.
2. Add type hints and docstrings, then run mypy.
3. Write a short maintainer guide in the project root.

### Progressive hints

1. Start with deterministic transformation/build functions. Create tiny
   synthetic DataFrames in fixtures so tests remain offline.
2. Type public boundaries first. A type-ignore requires a narrow reason;
   replacing every value with `Any` defeats the check.
3. Document setup, architecture, tests, formatting, data contracts, artifact
   locations, security rules, and review gates.

The separate solution demonstrates a small extraction, local pre-commit hooks,
and GitHub Actions. Treat CI as remote repetition of checks you can already run
locally.

### Additional mastery practice

Refactor behind tests, review risk at boundaries, and preserve behavior while improving structure. Type checks and formatting support—not replace—domain evidence.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Characterization testing:** Before refactoring a legacy notebook function, capture current behavior for normal, boundary, and known-bug inputs. Mark which behavior is a contract and which bug will intentionally change.
   **Progressive hint:** Characterization tests prevent accidental drift; an intentional fix needs a new expected result and a documented reason.
5. **Risk-based review:** Review a data-loading-to-prediction change using a checklist for security, data loss, leakage, schema compatibility, performance, error handling, and cross-platform paths.
   **Progressive hint:** Trace inputs to side effects and downstream consumers. Prioritize high-impact boundaries over cosmetic preferences.
6. **Compatibility change:** Rename a public function parameter without breaking callers. Implement a deprecation path, tests for old/new usage, and a removal plan.
   **Progressive hint:** Accept the old keyword temporarily, reject ambiguous double use, emit a targeted DeprecationWarning, and update docs/call sites.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Characterization testing


# Practice 5 — Risk-based review


# Practice 6 — Compatibility change
